# AII Cross-Source Consistency Check

The combined AII in `aii_quarterly.csv` blends three SEC filing types — 10-K (annual),
10-Q (quarterly), and 8-K (current reports) — into a single quarterly scalar.
Before using AII in predictive modeling we need to verify:

1. **Coverage**: Which document types appear in each quarter?
2. **Coherence**: Do per-source AII signals track each other (high cross-source correlation)?
3. **Dominance**: Does any single source dominate the combined signal?
4. **Divergence**: Are there quarters where sources tell different stories?

**Data model**
- `aii_quarterly.csv` — combined AII, one row per quarter (53 quarters, 2012-Q4 – 2025-Q4)
- `aii_by_doctype.csv` — per-source AII, one row per (quarter, doc_type) combination
  (92 rows; doc types: sec_10k, sec_10q, sec_8k)

Run `python3 -m measures.run_aii --dry-run` first to generate both CSV files.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from scipy.stats import pearsonr, spearmanr

DATA_DIR  = Path("../data/processed")
PLOTS_DIR = DATA_DIR / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

df_all = pd.read_csv(DATA_DIR / "aii_quarterly.csv")
df_all = df_all.sort_values(["year", "quarter"]).reset_index(drop=True)

df_src = pd.read_csv(DATA_DIR / "aii_by_doctype.csv")
df_src = df_src.sort_values(["year", "quarter", "doc_type"]).reset_index(drop=True)

print("Combined AII:", len(df_all), "quarters")
print("By-doctype AII:", len(df_src), "rows covering", df_src["doc_type"].nunique(), "source types")
print("Doc types:", sorted(df_src["doc_type"].unique()))

Combined AII: 53 quarters
By-doctype AII: 92 rows covering 3 source types
Doc types: ['sec_10k', 'sec_10q', 'sec_8k']


In [2]:
# ── Source coverage heatmap ───────────────────────────────────────────────────
# Pivot doc_count by (period, doc_type); align to the full 53-quarter range

coverage = df_src.pivot_table(index="period", columns="doc_type", values="doc_count", fill_value=0)
coverage = coverage.reindex(df_all["period"]).fillna(0).astype(int)
doc_types = sorted(coverage.columns)

fig, ax = plt.subplots(figsize=(4, 14))
data = coverage[doc_types].values
im = ax.imshow(data, aspect="auto", cmap="Blues", vmin=0)
plt.colorbar(im, ax=ax, label="doc_count", shrink=0.4)

ax.set_xticks(range(len(doc_types)))
ax.set_xticklabels(doc_types, rotation=45, ha="right", fontsize=9)
ax.set_yticks(range(len(coverage)))
ax.set_yticklabels(coverage.index, fontsize=6)
ax.set_title("Source Coverage\n(doc count per quarter)", fontsize=10)

out = PLOTS_DIR / "aii_cross_source_coverage.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

print("\nSource temporal coverage:")
for dt in doc_types:
    valid = coverage[dt][coverage[dt] > 0]
    if len(valid) > 0:
        print(f"  {dt}: {valid.index[0]} – {valid.index[-1]} ({len(valid)} quarters)")

Saved: ../data/processed/plots/aii_cross_source_coverage.png

Source temporal coverage:
  sec_10k: 2013-Q1 – 2025-Q1 (13 quarters)
  sec_10q: 2012-Q4 – 2025-Q4 (40 quarters)
  sec_8k: 2016-Q2 – 2025-Q4 (39 quarters)


In [3]:
# ── Z-score time series overlay ───────────────────────────────────────────────
# Wide-form pivot of AII by doc_type + combined; z-score per series

df_wide = df_src.pivot_table(index="period", columns="doc_type", values="aii")
df_wide = df_wide.reindex(df_all["period"])           # align to full 53-quarter range
df_wide["combined"] = df_all.set_index("period")["aii"].values

def zscore(s):
    valid = s.dropna()
    if len(valid) < 2 or valid.std() == 0:
        return s * 0
    return (s - valid.mean()) / valid.std()

df_z = df_wide.apply(zscore)

x = range(len(df_z))
labels = df_z.index.tolist()

COLORS = {"sec_10k": "steelblue", "sec_10q": "darkorange", "sec_8k": "mediumseagreen", "combined": "black"}
LW     = {"combined": 2.2}

gpt_idx = labels.index("2022-Q4") if "2022-Q4" in labels else len(labels)

fig, ax = plt.subplots(figsize=(14, 6))
ax.axvspan(0, gpt_idx, alpha=0.04, color="steelblue", label="Pre-GenAI era")
ax.axvspan(gpt_idx, len(labels), alpha=0.07, color="darkorange", label="Post-GenAI era")

for col in df_z.columns:
    ax.plot(
        x, df_z[col],
        label=col,
        linewidth=LW.get(col, 1.4),
        color=COLORS.get(col, "gray"),
        marker="o",
        markersize=3,
    )

ax.axhline(0, color="gray", linewidth=0.5, linestyle="--")
step = max(1, len(df_z) // 12)
ax.set_xticks([i for i in x if i % step == 0])
ax.set_xticklabels([labels[i] for i in x if i % step == 0], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("AII (z-score)", fontsize=11)
ax.set_title("Cross-Source AII — Z-Score Time Series", fontsize=13)
ax.legend(fontsize=9)

out = PLOTS_DIR / "aii_cross_source_zscore.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

Saved: ../data/processed/plots/aii_cross_source_zscore.png


In [4]:
# ── Pairwise correlation table ────────────────────────────────────────────────
# Pearson r and Spearman r for each pair over their shared non-NaN quarters

import warnings

series_dict = {col: df_wide[col].dropna() for col in df_wide.columns}
cols = list(series_dict.keys())

corr_rows = []
for i, c1 in enumerate(cols):
    for c2 in cols[i + 1:]:
        idx = series_dict[c1].index.intersection(series_dict[c2].index)
        if len(idx) < 5:
            continue
        s1, s2 = series_dict[c1][idx], series_dict[c2][idx]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")   # suppress ConstantInputWarning for zero-only series
            pr, _ = pearsonr(s1, s2)
            sr, _ = spearmanr(s1, s2)
        corr_rows.append({
            "source_1":   c1,
            "source_2":   c2,
            "n_quarters": len(idx),
            "pearson_r":  round(pr, 3) if not np.isnan(pr) else float("nan"),
            "spearman_r": round(sr, 3) if not np.isnan(sr) else float("nan"),
        })

corr_df = pd.DataFrame(corr_rows).sort_values("pearson_r", ascending=False, na_position="last")
print("Pairwise correlation (over shared non-NaN quarters):")
print(corr_df.to_string(index=False))

Pairwise correlation (over shared non-NaN quarters):
source_1 source_2  n_quarters  pearson_r  spearman_r
 sec_10q combined          40      0.998       0.996
 sec_10k combined          13      0.997       0.994
  sec_8k combined          39      0.185       0.238
 sec_10q   sec_8k          30      0.174       0.243
 sec_10k   sec_8k           9        NaN         NaN


In [5]:
# ── Divergence analysis ───────────────────────────────────────────────────────
# Per-quarter AII range across doc types (max − min); quarters with ≥2 sources

from matplotlib.patches import Patch

src_only = df_wide.drop(columns=["combined"])
n_valid  = src_only.notna().sum(axis=1)

div_max = src_only.max(axis=1, skipna=True)
div_min = src_only.min(axis=1, skipna=True)
div     = (div_max - div_min).where(n_valid >= 2, 0.0).fillna(0.0)

top10 = div.sort_values(ascending=False).head(10)
print("Top-10 highest-divergence quarters:")
print(top10.round(4).to_string())

def era(period: str) -> str:
    year = int(period.split("-")[0])
    q    = int(period[-1])
    return "Post-GenAI" if (year > 2022 or (year == 2022 and q >= 4)) else "Pre-GenAI"

x_div      = range(len(div))
labels_div = div.index.tolist()
era_colors = ["darkorange" if era(p) == "Post-GenAI" else "steelblue" for p in labels_div]

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x_div, div.values, color=era_colors, alpha=0.8)

step_div = max(1, len(div) // 12)
ax.set_xticks([i for i in x_div if i % step_div == 0])
ax.set_xticklabels(
    [labels_div[i] for i in x_div if i % step_div == 0],
    rotation=45, ha="right", fontsize=8,
)
ax.set_ylabel("AII range (max − min across sources)", fontsize=11)
ax.set_title("Cross-Source AII Divergence by Quarter", fontsize=13)
ax.legend(
    handles=[Patch(color="steelblue", label="Pre-GenAI"), Patch(color="darkorange", label="Post-GenAI")],
    fontsize=9,
)

out = PLOTS_DIR / "aii_cross_source_divergence.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"\nSaved: {out}")

Top-10 highest-divergence quarters:
period
2019-Q4    0.6265
2020-Q1    0.3180
2025-Q1    0.2853
2022-Q4    0.2407
2020-Q2    0.2339
2025-Q2    0.2174
2020-Q3    0.2167
2024-Q1    0.2152
2025-Q4    0.2133
2020-Q4    0.2061

Saved: ../data/processed/plots/aii_cross_source_divergence.png


In [6]:
# ── Source contribution breakdown ─────────────────────────────────────────────
# Each doc_type's share of the quarter's combined raw score

merged = df_src.merge(
    df_all[["period", "quarter_raw_score"]].rename(columns={"quarter_raw_score": "total_raw"}),
    on="period",
)
merged["share"] = merged["quarter_raw_score"] / merged["total_raw"].replace(0, np.nan)

share_wide = merged.pivot_table(index="period", columns="doc_type", values="share", fill_value=0)
share_wide = share_wide.reindex(df_all["period"]).fillna(0)
doc_types_sorted = sorted(share_wide.columns)

CONTRIB_COLORS = {"sec_10k": "steelblue", "sec_10q": "darkorange", "sec_8k": "mediumseagreen"}

x_sh      = range(len(share_wide))
labels_sh = share_wide.index.tolist()
bottom    = np.zeros(len(share_wide))

fig, ax = plt.subplots(figsize=(14, 5))
for dt in doc_types_sorted:
    vals = share_wide[dt].values
    ax.bar(x_sh, vals, bottom=bottom, label=dt, color=CONTRIB_COLORS.get(dt, "gray"), alpha=0.85)
    bottom += vals

step_sh = max(1, len(share_wide) // 12)
ax.set_xticks([i for i in x_sh if i % step_sh == 0])
ax.set_xticklabels(
    [labels_sh[i] for i in x_sh if i % step_sh == 0],
    rotation=45, ha="right", fontsize=8,
)
ax.set_ylabel("Share of quarter raw score", fontsize=11)
ax.set_title("Source Contribution to Quarterly Raw Score", fontsize=13)
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))

out = PLOTS_DIR / "aii_cross_source_contribution.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {out}")

# Sanity check: shares sum to ~1.0 for quarters with data
row_sums = share_wide.sum(axis=1)
nonzero  = row_sums[row_sums > 0]
print(f"\nShare sum check — min: {nonzero.min():.4f}, max: {nonzero.max():.4f} (expect ~1.0 for all nonzero quarters)")

Saved: ../data/processed/plots/aii_cross_source_contribution.png

Share sum check — min: 1.0000, max: 1.0000 (expect ~1.0 for all nonzero quarters)


## Interpretation

### Corpus composition (canonical_v2)
The canonical_v2 corpus contains three SEC filing types:
- **sec_10k** (annual): 13 quarters — Q1 of each fiscal year 2013–2025. Covers the full time span with one filing per year.
- **sec_10q** (quarterly): 40 quarters — dense Q1–Q4 coverage starting 2012-Q4.
- **sec_8k** (current reports): 39 quarters — present from 2017-Q3 onward, 1–4 docs per quarter.

### Cross-source coherence
The z-score overlay and pairwise correlations reveal whether sources agree on the
direction and magnitude of AI language intensity changes. A high cross-source
Pearson r (> 0.7) would validate the combined AII as a robust signal; a low r
would suggest the composite is not coherent and source-specific signals may be needed.

### Source dominance in peak quarters
The contribution chart shows which filing type drives the highest-AII quarters
(2020-Q1, 2022-Q4, 2025-Q1). If a single source consistently accounts for > 80%
of raw score in peak quarters, the combined AII is effectively a proxy for that
source alone.

### Divergence events
High-divergence quarters (large max − min across sources) indicate that different
document types contain meaningfully different AI language intensity. These events
are candidates for deeper analysis — e.g., did 8-K filings spike around a product
announcement while 10-Q filings remained flat?

### Modeling recommendation
If cross-source correlation is high (r > 0.7) and no single source dominates,
the combined AII is appropriate as a single predictive feature. If sources diverge
systematically, consider using source-specific AII columns as separate model inputs
or constructing a source-weighted composite that accounts for the reliability of
each filing type.